# Combined Amber RAG (ChromaDB archive + PDF FAISS) — POC

Single query → hits **two** vector stores in parallel, merges results by similarity score, keeps the top 5 overall, feeds them to the LLM.

- **Store A** — persistent **Chroma** at `/opt/chromadb/data/prompt_db`.
- **Store B** — in-memory **FAISS** built once from `Amber25.pdf`, cached to `./faiss_pdf_index/` so it loads instantly on subsequent runs.

Both stores use `sentence-transformers/all-MiniLM-L6-v2` and L2 (squared) distance, so scores are directly comparable across stores.

## 1) Setup

In [2]:
import os
from typing import List, Tuple

import chromadb
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_ollama import ChatOllama

## 2) Shared embedding model
Both stores must use the same embedding model for the scores to be comparable.

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## 3) Open both stores

- **Archive store** — persistent Chroma at `/opt/chromadb/data/prompt_db`.
- **PDF store** — in-memory FAISS. On first run, it parses `Amber25.pdf`, embeds the chunks, and saves a FAISS index to `./faiss_pdf_index/`. On every subsequent run, it loads that cached index in < 1s instead of re-embedding ~3,800 chunks.

Delete `./faiss_pdf_index/` to force a rebuild (e.g. when the PDF or chunking settings change).

In [4]:
# --- Store A: archive / emails (persistent Chroma) ---
ARCHIVE_DB_PATH = "/opt/chromadb/data/prompt_db"
archive_client = chromadb.PersistentClient(path=ARCHIVE_DB_PATH)
vectorstore_archive = Chroma(
    client=archive_client,
    collection_name="rag_collection",
    embedding_function=embeddings,
)

# --- Store B: PDF (in-memory FAISS with cached index on disk) ---
PDF_PATH = "Amber25.pdf"
FAISS_INDEX_DIR = "./faiss_pdf_index"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100


def _clean_pdf_text(text: str) -> str:
    text = " ".join(text.split())
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    return text


def _build_pdf_chunks(pdf_path: str) -> List[Document]:
    pages = PyPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=[" "]
    )
    chunks: List[Document] = []
    for page_num, page in enumerate(pages):
        cleaned = _clean_pdf_text(page.page_content)
        if len(cleaned.strip()) < 50:
            continue
        chunks.extend(splitter.create_documents(
            texts=[cleaned],
            metadatas=[{
                **page.metadata,
                "page": page_num + 1,
                "total_pages": len(pages),
                "chunk_method": "smart_pdf_processor",
                "char_count": len(cleaned),
            }],
        ))
    return chunks


if os.path.isdir(FAISS_INDEX_DIR):
    vectorstore_pdf = FAISS.load_local(
        FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded cached FAISS index from {FAISS_INDEX_DIR}")
else:
    print(f"No cached index at {FAISS_INDEX_DIR} — building from {PDF_PATH} ...")
    pdf_chunks = _build_pdf_chunks(PDF_PATH)
    print(f"  {len(pdf_chunks)} chunks, embedding ...")
    vectorstore_pdf = FAISS.from_documents(pdf_chunks, embeddings)
    vectorstore_pdf.save_local(FAISS_INDEX_DIR)
    print(f"  Saved FAISS index to {FAISS_INDEX_DIR}")

print(f"Archive store: {vectorstore_archive._collection.count()} vectors @ {ARCHIVE_DB_PATH}")
print(f"PDF store:     {vectorstore_pdf.index.ntotal} vectors (FAISS in-memory)")

Loaded cached FAISS index from ./faiss_pdf_index
Archive store: 0 vectors @ /opt/chromadb/data/prompt_db
PDF store:     3793 vectors (FAISS in-memory)


/tmp/ipykernel_399157/1820872167.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore_archive = Chroma(


## 4) Hybrid retriever — merge both stores, keep top 5
Each store is queried for its own top-K. We tag every doc with which store it came from, sort by raw L2 distance (lower = closer), and keep the best `k_total` overall.

In [5]:
def hybrid_retrieve(
    query: str,
    k_total: int = 5,
    k_archive: int = 10,
    k_pdf: int = 10,
) -> List[Document]:
    """Query both stores, merge by distance (lower = closer), return top k_total docs."""
    scored: List[Tuple[Document, float]] = []

    for doc, score in vectorstore_archive.similarity_search_with_score(query, k=k_archive):
        doc.metadata = {**doc.metadata, "store": "archive", "score": float(score)}
        scored.append((doc, score))

    for doc, score in vectorstore_pdf.similarity_search_with_score(query, k=k_pdf):
        doc.metadata = {**doc.metadata, "store": "pdf", "score": float(score)}
        scored.append((doc, score))

    scored.sort(key=lambda pair: pair[1])  # L2: lower is closer
    return [doc for doc, _ in scored[:k_total]]

# Quick smoke test
hits = hybrid_retrieve("What is Amber?", k_total=5)
for i, d in enumerate(hits, 1):
    print(f"{i}. [{d.metadata.get('store')}] score={d.metadata.get('score'):.4f} "
          f"page={d.metadata.get('page', '-')}")
    print("   ", d.page_content[:160].replace("\n", " "), "...")


1. [pdf] score=0.7695 page=1
    Amber 2025 Reference Manual (Covers Amber24 and AmberTools25) ...
2. [pdf] score=0.7889 page=262
    used to identify ATOMs. type This is a STRING property that defines the AMBER force field atom type. charge The charge property is a NUMBER that represents the  ...
3. [pdf] score=0.8482 page=15
    1. Introduction Amber is the collective name for a suite of programs that allow users to carry out molecular dynamics simu- lations, particularly on biomolecule ...
4. [pdf] score=0.9081 page=24
    option, though it does not control the install directory. For many libraries which are required and are not commonly found on people’s systems, Amber provides b ...
5. [pdf] score=0.9266 page=27
    2.3. Applying Updates 2.3.1. Basic Usage Updates to AmberTools and Amber are downloaded, applied, and managed automatically using the Python script update amber ...


## 5) LLM + prompt

In [6]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)

In [7]:
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).

CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention any Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output the final answer and Include citations, references, or metadata.

Context:
{context}

Question: {question}

Answer:""")

## 6) LCEL chain using the hybrid retriever

In [8]:
def format_docs(docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "unknown")
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "-")
        score = doc.metadata.get("score", None)
        score_str = f" | Score: {score:.4f}" if isinstance(score, (int, float)) else ""
        formatted.append(
            f"[Chunk {i} | Store: {store} | Source: {source} | Page: {page}{score_str}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [9]:
# Wrap the hybrid retrieval in a RunnableLambda so it plugs into LCEL like any other retriever.
hybrid_retriever = RunnableLambda(lambda q: hybrid_retrieve(q, k_total=5))

rag_chain_lcel = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

## 7) Query helper — answer + the top-5 sources that fed it

In [10]:
def query_rag(question: str, k_total: int = 5):
    print(f"Question: {question}")
    print("-" * 60)

    docs = hybrid_retrieve(question, k_total=k_total)
    answer = rag_chain_lcel.invoke(question)

    print("Answer:")
    print(answer)

    print("\nTop sources used:")
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "?")
        page = doc.metadata.get("page", "-")
        src = doc.metadata.get("source", "-")
        score = doc.metadata.get("score")
        score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "-"
        print(f"\n--- Source {i} [{store}] page={page} src={src} score={score_str} ---")
        print(doc.page_content[:240].replace("\n", " "), "...")

    return answer, docs

## 8) Try it

In [11]:
_ = query_rag("What is Amber?")

Question: What is Amber?
------------------------------------------------------------
Answer:
Amber is the collective name for a suite of programs that allow users to carry out molecular dynamics simulations, particularly on biomolecules.

Technical Explanation:
The term "Amber" refers to both the software suite and the empirical force fields implemented within it. The code and force field are separate entities, with the code distributed under a license agreement and the force fields in the public domain. The Amber software suite is divided into two parts: AmberTools25 and Amber24.

Practical Guidance:
To use the Amber programs, users can download and install the software from the official website. Updates to AmberTools and Amber are managed automatically using the Python script `update amber`. Users can query for updates and check their version using the `$AMBERSOURCE/update amber` command.

Reference: [1-4]

Top sources used:

--- Source 1 [pdf] page=1 src=Amber25.pdf score=0.7695 --

In [12]:
_ = query_rag("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
------------------------------------------------------------
Answer:
The minimizer is not aware of what SHAKE is doing; for this reason, minimizations generally should be carried out without SHAKE.

Technical Explanation:
This is because SHAKE is an algorithm based on dynamics, and the minimizer does not have knowledge about the constraints applied by SHAKE. As a result, it's recommended to disable SHAKE during minimization to ensure accurate results.

Practical Guidance:
To disable SHAKE during minimization in AMBER, set the SHAKE flag to 1 (default) or use the &cntrl namelist variable ntc = 0.

Top sources used:

--- Source 1 [pdf] page=20 src=Amber25.pdf score=0.9120 ---
and MAX gradients reported in sander are often more precision sensitive than the energies, and may vary by 1 in the last figure on some machines. In minimization and dynamics calculations, it is not unusual to see small divergences in behav ...

--

In [13]:
_ = query_rag("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
------------------------------------------------------------
Answer:
To obtain a Z-DNA structure from NAB, you can use the `cpptraj` command with the `nastruct` keyword. 

First, determine base pairing using either the first frame or a reference structure, or specify it via the `specifiedbp` and `pairs` keywords.

Then, run `cpptraj` with the following command:
```bash
cpptraj naout nastruct.dat resrange 1-3,28-30 run writedata NApucker.dat NA[pucker]
```
This will generate a Z-DNA structure. Note that base pair data sets are not created until base pairing is determined.

Technical Explanation:

The `nastruct` keyword in `cpptraj` calculates basic nucleic acid (NA) structure parameters for all residues in the specified range. The procedure used to calculate NA structural parameters is the same as 3DNA, with algorithms adapted from Babcok et al. and reference frame coordinates from Olson et al.

The `resrange` keyword specifies the r

In [14]:
_ = query_rag("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
------------------------------------------------------------
Answer:
To get SHAKE to consider two different residue names to be water, you need to redefine the default residue name used by the program to determine which residues are waters. You can do this by setting the `WATNAM` variable.

According to Chunk 1 of the provided Context (page 406), the default residue name for water is 'W AT'. To use a different residue name, you need to redefine it using the `WATNAM` variable. For example, if you want to consider residues named 'H2O' as water, you would set `WATNAM = H2O`.

Technical Explanation:
The SHAKE algorithm is used to constrain bonds involving hydrogen atoms in molecular dynamics simulations. By default, the program searches for water residues and uses special routines to SHAKE these systems. However, if you want to use a different residue name as water, you need to redefine the `WATNAM` variable

In [15]:
_ = query_rag("Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?")

Question: Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?
------------------------------------------------------------
Answer:
Based on the provided Context, we can infer that:

1. **Rocky Linux and Ubuntu are both supported**: Although not explicitly mentioned in the Context, it's reasonable to assume that both Rocky Linux and Ubuntu are supported operating systems for installing Amber + CUDA.
2. **CUDA version compatibility is crucial**: The Context emphasizes the importance of using newer CUDA versions (7.5 to 12.6 inclusive) for optimal performance and stability.
3. **Environment variable setup is essential**: The environment variable `CUDA_HOME` should be set to point to your NVIDIA Toolkit installation, and `$CUDAHOME/bin/` should be in your path.

Given these considerations, we can provide a recommendation:

**Best OS recommendation: Ubuntu**

Ubuntu is a widely used and well-documented Linux distribut

In [16]:
_ = query_rag("How do I use paramfit to generate force field parameters for boron-containing compounds?")

Question: How do I use paramfit to generate force field parameters for boron-containing compounds?
------------------------------------------------------------
Answer:
Unfortunately, the provided Context does not contain information on how to use paramfit to generate force field parameters specifically for boron-containing compounds. However, it does provide general information on using paramfit and generating force field parameters.

To answer your question, I will rely on the general guidance provided in the Context:

1. Paramfit can be used when existing parameters do not describe the system to the desired level of accuracy.
2. Paramfit attempts to fit the AMBER energy to the quantum energy for a variety of conformations of the input structure.

Since boron-containing compounds are not mentioned specifically, I will provide general guidance on using paramfit:

To use paramfit, you need to prepare your input files and specify which parameters to fit. You can use the `paramfit` progra

In [17]:
_ = query_rag("I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?")

Question: I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?
------------------------------------------------------------
Answer:
To extract PDB files for the two states, follow these steps:

1. **Parameterize the molecule as usual**: This includes adding dummy atoms if necessary.
2. **Create a PDB file with both molecules separated by a TER card**: Ensure that V0 and V1 have the same number of atoms (Chunk 1).
3. **Update residue numbers for the second molecule**: Use different residue names for each state, as they are distinct (Chunk 4).

For extracting the PDB files:

* Run `cpptraj` with the following input:
```bash
cpptraj -i cpptraj.in
```
where `cpptraj.in` contains the following lines:
```amber
trajin pmemd.trj1
trajout stateA.pdb first 0 last 10

In [18]:
_ = query_rag("""
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?
""")

Question: 
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?

------------------------------------------------------------
Answer:
The correct way to identify parallel strand DNA in CPPTRAJ's nastruct command is by specifying the base pair type using the `bptype` option.

According to the manual, when you run `guessbp bptype para`, it should correctly identify the molecule as parallel strands. However, if it gets stuck or appears not to continue running, try checking the following:

1. Ensure that the input topology file is correct and contains the necessary information for nastruct to iden